# FPLedge Backtest — Model vs Real Season

Compare model decisions vs your real 2025/26 FPL season, GW-by-GW.

**Setup before running:**
1. Generate model results: `python scripts/backtest_season.py --use-engine --initial-squad data/backtest/my_gw1_squad.csv --chips --can-bonus --out data/backtest/results_full.csv`
2. Edit `MY_REAL_SCORES` below with your actual GW points (from your FPL history page)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Load backtest results

In [ ]:
results_path = Path('../data/backtest/results_full.csv')
results = pd.read_csv(results_path)
print(f'Loaded: {len(results)} GWs from {results_path}')
results.head()

## Enter your real GW points

From your FPL history at https://fantasy.premierleague.com/entry/<your_id>/history.

Replace the placeholder list with your actual GW1-29 points. The example below sums to 1689 at GW29.

In [ ]:
# Your real points per GW. GW1 first.
# Replace these with your actual scores from FPL history.
MY_REAL_SCORES = {
    1: 65,   # placeholder — REPLACE with your actual GW1 score
    2: 50,
    3: 55,
    4: 60,
    5: 48,
    6: 52,
    7: 58,
    8: 70,
    9: 55,
    10: 65,
    11: 45,
    12: 50,
    13: 60,
    14: 55,
    15: 70,
    16: 80,   # CAN bonus week — likely a big haul
    17: 58,
    18: 50,
    19: 65,
    20: 55,
    21: 60,
    22: 70,
    23: 55,
    24: 65,
    25: 60,
    26: 75,   # likely your TC/BB week if used here
    27: 55,
    28: 60,
    29: 64,
}

your_total = sum(MY_REAL_SCORES.values())
print(f'Your total entered: {your_total} pts (should match your real GW29 score)')

real_df = pd.DataFrame({
    'gw': list(MY_REAL_SCORES.keys()),
    'your_points': list(MY_REAL_SCORES.values()),
})
real_df['your_total'] = real_df['your_points'].cumsum()

## Merge model vs you

In [ ]:
merged = results.merge(real_df, on='gw', how='outer').sort_values('gw').reset_index(drop=True)
merged['model_points'] = merged['points']
merged['model_total'] = merged['total']
merged['gw_delta'] = merged['model_points'] - merged['your_points']
merged['cum_delta'] = merged['model_total'] - merged['your_total']
merged

## Summary

In [ ]:
final_model = merged['model_total'].dropna().iloc[-1]
final_you = merged['your_total'].dropna().iloc[-1]
gap = final_model - final_you

print(f'Final score after GW{int(merged.gw.max())}:')
print(f'  Model: {final_model:.0f} pts')
print(f'  You:   {final_you:.0f} pts')
print(f'  Gap:   {gap:+.0f} pts')

print(f'\nModel beat you in {(merged["gw_delta"] > 0).sum()} of {len(merged)} GWs')
print(f'Your best GW: {merged.loc[merged["your_points"].idxmax(), "gw"]:.0f} ({merged["your_points"].max():.0f} pts)')
print(f'Model best GW: {merged.loc[merged["model_points"].idxmax(), "gw"]:.0f} ({merged["model_points"].max():.0f} pts)')

## Per-GW comparison chart

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Cumulative totals
axes[0].plot(merged['gw'], merged['model_total'], 'o-', label='Model', color='#e91e63', linewidth=2)
axes[0].plot(merged['gw'], merged['your_total'], 's-', label='You', color='#2196f3', linewidth=2)
axes[0].set_ylabel('Cumulative points')
axes[0].set_title('Cumulative score: Model vs You')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Per-GW points (bars side by side)
x = merged['gw']
w = 0.4
axes[1].bar(x - w/2, merged['model_points'], w, label='Model', color='#e91e63', alpha=0.8)
axes[1].bar(x + w/2, merged['your_points'], w, label='You', color='#2196f3', alpha=0.8)
axes[1].set_xlabel('Gameweek')
axes[1].set_ylabel('Points')
axes[1].set_title('Per-GW score')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Mark chip GWs
if 'chip' in merged.columns:
    chip_rows = merged[merged['chip'].fillna('') != '']
    for _, r in chip_rows.iterrows():
        axes[1].annotate(r['chip'][:3].upper(), (r['gw'], r['model_points']),
                          textcoords='offset points', xytext=(0, 5), ha='center', fontsize=8, color='#e91e63')

plt.tight_layout()
plt.show()

## Captain comparison

Each GW the model picked these captains.

In [ ]:
cap_summary = results[['gw', 'captain', 'captain_pts', 'chip']].copy()
cap_summary['captain_pts_x2'] = cap_summary['captain_pts'] * 2
cap_summary.loc[cap_summary['chip'] == 'triple_captain', 'captain_pts_x2'] = cap_summary['captain_pts'] * 3
cap_summary

## Captain hit rate

In [ ]:
hit_rate_10 = (results['captain_pts'] >= 10).mean() * 100
hit_rate_15 = (results['captain_pts'] >= 15).mean() * 100
haul_rate = (results['captain_pts'] >= 6).mean() * 100
blank_rate = (results['captain_pts'] <= 2).mean() * 100

print(f'Captain blanks (≤2 pts):    {blank_rate:.0f}%')
print(f'Captain plays (3-5 pts):    {100 - blank_rate - haul_rate:.0f}%')
print(f'Captain haul (≥6 pts):       {haul_rate:.0f}%')
print(f'Captain big haul (≥10 pts):  {hit_rate_10:.0f}%')
print(f'Captain massive (≥15 pts):   {hit_rate_15:.0f}%')

## Transfer activity

In [ ]:
transfers = results[results['transfer_in'].fillna('') != ''].copy()
print(f'Total transfers: {len(transfers)}')
print(f'Total hits: -{results["hit"].sum():.0f} pts')
print(f'Avg transfers per GW: {len(transfers) / len(results):.2f}')

# Most-transferred-in players
in_counts = transfers['transfer_in'].value_counts().head(10)
out_counts = transfers['transfer_out'].value_counts().head(10)

print('\nMost transferred IN:')
print(in_counts)
print('\nMost transferred OUT:')
print(out_counts)

## Where did the model lose / win points?

In [ ]:
merged_valid = merged.dropna(subset=['model_points', 'your_points'])

print('Worst GWs for model (vs you):')
print(merged_valid.nsmallest(5, 'gw_delta')[['gw', 'model_points', 'your_points', 'gw_delta', 'captain', 'chip']])

print('\nBest GWs for model (vs you):')
print(merged_valid.nlargest(5, 'gw_delta')[['gw', 'model_points', 'your_points', 'gw_delta', 'captain', 'chip']])